# EDA: Heart Disease Playground

Purpose: move through the requested plan Phases 1-4 via the shortest path.
Target data: `data/raw/train.csv`, `data/raw/test.csv`, `data/raw/sample_submission.csv`


### Package import

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Prefer Kaggle input paths if available; fallback to local data/raw.
kaggle_dir = Path("/kaggle/input/playground-series-s6e2")
if kaggle_dir.exists():
    train = pd.read_csv(kaggle_dir / "train.csv")
    test = pd.read_csv(kaggle_dir / "test.csv")
    sub = pd.read_csv(kaggle_dir / "sample_submission.csv")
else:
    # Infer project root (so it works even when run from nb/)
    cwd = Path.cwd().resolve()
    project_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "data" / "raw" / "train.csv").exists():
            project_root = p
            break
    if project_root is None:
        raise FileNotFoundError("project root not found: data/raw/train.csv")

    data_dir = project_root / "data" / "raw"
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sub = pd.read_csv(data_dir / "sample_submission.csv")

print(train.shape, test.shape, sub.shape)


### Data download

In [ ]:
display(train.head())
display(test.head())
display(sub.head())

In [ ]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


### Data preprocessing

In [ ]:
id_col = "id" if "id" in train.columns else None
if id_col:
    print("train dup id:", train[id_col].duplicated().sum())
    print("test dup id:", test[id_col].duplicated().sum())
    train_ids = set(train[id_col])
    test_ids = set(test[id_col])
    print("train/test overlap:", len(train_ids & test_ids))
else:
    print("No id column found.")


In [ ]:
def missing_zero_table(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().mean().rename("missing_rate")
    uniq = df.nunique(dropna=True).rename("n_unique")
    dtype = df.dtypes.rename("dtype")

    num_cols_ = df.select_dtypes(include=["number"]).columns
    zero_rate = pd.Series(index=df.columns, dtype="float64", name="zero_rate")
    zero_rate.loc[num_cols_] = (df[num_cols_] == 0).mean()

    out = pd.concat([dtype, uniq, missing, zero_rate], axis=1)
    return out.sort_values(["missing_rate", "zero_rate"], ascending=False)

mz = missing_zero_table(train)
mz.head(30)


### Data labeling

In [ ]:
# Infer target column
target_col = None
for cand in ["Heart Disease", "HeartDisease", "target", "label"]:
    if cand in train.columns:
        target_col = cand
        break
print("target_col:", target_col)

if target_col:
    display(train[target_col].value_counts(dropna=False))
    display(train[target_col].value_counts(normalize=True, dropna=False))
    print("dtype:", train[target_col].dtype)


In [ ]:
TARGET_RAW = "Heart Disease"
target_map = {"Absence": 0, "Presence": 1}

y = train[TARGET_RAW].map(target_map)
assert y.isna().sum() == 0

print("pos_rate:", float(y.mean()))

ID_COL = "id"
FEATURE_COLS = [c for c in train.columns if c not in [target_col, ID_COL]]

cat_cols = [c for c in FEATURE_COLS if train[c].nunique(dropna=True) <= 20]
num_cols = [
    c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(train[c]) and c not in cat_cols
]

for c in cat_cols:
    train[c] = train[c].astype("category")
    test[c] = test[c].astype("category")


### Categorical candidates

In [ ]:
# Numeric columns with few uniques are categorical candidates
n_unique = train[FEATURE_COLS].nunique(dropna=True).sort_values()
small_unique = n_unique[n_unique <= 20]
small_unique


### Numeric summary

In [ ]:
train[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


### Relationship to target

- Numeric: mean/median by target
- Categorical: target rate by category


In [ ]:
if target_col:
    # Numeric
    if len(num_cols) > 0:
        display(train.assign(y=y).groupby("y")[num_cols].agg(["mean", "median"]).T)

    def pos_rate_by_category(col: str) -> pd.DataFrame:
        tmp = pd.DataFrame({"x": train[col], "y": y})
        return (
            tmp.groupby("x")["y"]
            .agg(pos_rate="mean", count="size")
            .sort_values(["pos_rate", "count"], ascending=[False, False])
        )

    for c in cat_cols:
        display(pos_rate_by_category(c))


In [ ]:
for c in num_cols:
    if (train[c] == 0).any():
        rate0 = y[train[c] == 0].mean()
        rate1 = y[train[c] != 0].mean()
        print(
            f"{c:25s} zero_rate={(train[c]==0).mean():.3f}  pos_rate(0)={rate0:.3f}  pos_rate(!0)={rate1:.3f}"
        )


In [ ]:
zero_sensitive = ["ST depression", "Number of vessels fluro"]
for c in zero_sensitive:
    train[f"{c}__is_zero"] = (train[c] == 0).astype("int8")
    test[f"{c}__is_zero"] = (test[c] == 0).astype("int8")


In [ ]:
# Numeric: target-wise hist (sampled for speed)
cols = ["Age", "Max HR", "ST depression"]
tmp = train.sample(50_000, random_state=42).assign(y=y)

for c in cols:
    plt.figure()
    tmp[tmp["y"] == 0][c].hist(alpha=0.5, bins=30)
    tmp[tmp["y"] == 1][c].hist(alpha=0.5, bins=30)
    plt.title(c)
    plt.show()


In [ ]:
# Categorical: pos_rate by category
cat_plot_cols = ["Chest pain type", "Thallium", "Number of vessels fluro"]

for c in cat_plot_cols:
    if c not in train.columns:
        continue
    df = pos_rate_by_category(c)
    plt.figure(figsize=(6, 3))
    df["pos_rate"].plot(kind="bar")
    plt.title(c)
    plt.ylabel("pos_rate")
    plt.tight_layout()
    plt.show()


### Train vs Test distribution drift

- Adversarial validation (CatBoost, train-vs-test AUC)
- Categorical: rate differences


In [ ]:
from catboost import CatBoostClassifier, CatBoostError
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

use_gpu = True

adv_cols = [c for c in FEATURE_COLS if c in test.columns]
adv_cat_cols = [c for c in adv_cols if train[c].nunique(dropna=True) <= 20]

X_adv = pd.concat([
    train[adv_cols].assign(_is_test=0),
    test[adv_cols].assign(_is_test=1),
], axis=0).reset_index(drop=True)
y_adv = X_adv.pop("_is_test")

X_adv_train, X_adv_valid, y_adv_train, y_adv_valid = train_test_split(
    X_adv,
    y_adv,
    test_size=0.2,
    random_state=42,
    stratify=y_adv,
)

adv_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    od_type="Iter",
    od_wait=150,
    depth=5,
    learning_rate=0.08,
    random_seed=42,
    verbose=200,
)
if use_gpu:
    adv_params.update(task_type="GPU", devices="0")
else:
    adv_params.update(task_type="CPU")

adv_model = CatBoostClassifier(**adv_params)

try:
    adv_model.fit(
        X_adv_train,
        y_adv_train,
        cat_features=adv_cat_cols,
        eval_set=(X_adv_valid, y_adv_valid),
        use_best_model=True,
    )
except CatBoostError:
    # GPU can be unavailable on some environments; keep a CPU fallback.
    cpu_params = {k: v for k, v in adv_params.items() if k not in ["task_type", "devices"]}
    cpu_params["task_type"] = "CPU"
    adv_model = CatBoostClassifier(**cpu_params)
    adv_model.fit(
        X_adv_train,
        y_adv_train,
        cat_features=adv_cat_cols,
        eval_set=(X_adv_valid, y_adv_valid),
        use_best_model=True,
    )

adv_valid_pred = adv_model.predict_proba(X_adv_valid)[:, 1]
adv_auc = roc_auc_score(y_adv_valid, adv_valid_pred)
print(f"Adversarial validation AUC (train vs test): {adv_auc:.4f}")

drift_num = (
    pd.DataFrame(
        {
            "col": adv_cols,
            "adversarial_importance": adv_model.get_feature_importance(),
        }
    )
    .sort_values("adversarial_importance", ascending=False)
    .reset_index(drop=True)
)
drift_num.head(20)


### Categorical rate differences

In [ ]:
cat_cols = [c for c in FEATURE_COLS if c in test.columns and train[c].nunique(dropna=True) <= 20]

rows = []
for c in cat_cols:
    train_rate = train[c].value_counts(normalize=True)
    test_rate = test[c].value_counts(normalize=True)
    diff = (train_rate - test_rate).abs().fillna(0).sum() / 2
    rows.append((c, diff))

drift_cat = pd.DataFrame(rows, columns=["col", "total_variation_distance"]).sort_values(
    "total_variation_distance", ascending=False
)
drift_cat.head(20)


In [ ]:
from catboost import CatBoostClassifier, CatBoostError
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

use_gpu = bool(globals().get("use_gpu", True))

model_feature_cols = [c for c in train.columns if c not in [target_col, ID_COL]]
cat_cols_model = [c for c in model_feature_cols if train[c].nunique(dropna=True) <= 20]

X_model = train[model_feature_cols].copy()

train_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    od_type="Iter",
    od_wait=150,
    depth=5,
    learning_rate=0.08,
    random_seed=42,
    verbose=False,
)
if use_gpu:
    train_params.update(task_type="GPU", devices="0")
else:
    train_params.update(task_type="CPU")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_model, y), start=1):
    X_tr, X_va = X_model.iloc[tr_idx], X_model.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    fold_model = CatBoostClassifier(**train_params)
    try:
        fold_model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols_model,
            eval_set=(X_va, y_va),
            use_best_model=True,
            verbose=False,
        )
    except CatBoostError:
        cpu_params = {k: v for k, v in train_params.items() if k not in ["task_type", "devices"]}
        cpu_params["task_type"] = "CPU"
        fold_model = CatBoostClassifier(**cpu_params)
        fold_model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols_model,
            eval_set=(X_va, y_va),
            use_best_model=True,
            verbose=False,
        )

    va_pred = fold_model.predict_proba(X_va)[:, 1]
    rows.append(
        {
            "fold": fold,
            "roc_auc": roc_auc_score(y_va, va_pred),
            "best_iteration": fold_model.get_best_iteration(),
        }
    )

cv_table = pd.DataFrame(rows)
cv_table


In [ ]:
summary = pd.DataFrame(
    {
        "metric": ["mean_roc_auc", "std_roc_auc", "mean_best_iteration"],
        "value": [
            cv_table["roc_auc"].mean(),
            cv_table["roc_auc"].std(),
            cv_table["best_iteration"].mean(),
        ],
    }
)
summary


### Submit prediction

In [ ]:
from catboost import CatBoostClassifier, CatBoostError
from sklearn.model_selection import train_test_split
import pandas as pd

use_gpu = bool(globals().get("use_gpu", True))

TARGET = target_col
ID_COL = "id"

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
cat_cols = [c for c in feature_cols if train[c].nunique(dropna=True) <= 20]

X_train_full = train[feature_cols].copy()
X_test = test[feature_cols].copy()

X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train_full,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y,
)

fit_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    od_type="Iter",
    od_wait=150,
    depth=5,
    learning_rate=0.08,
    random_seed=42,
    verbose=False,
)
if use_gpu:
    fit_params.update(task_type="GPU", devices="0")
else:
    fit_params.update(task_type="CPU")

model_es = CatBoostClassifier(**fit_params)

try:
    model_es.fit(
        X_fit,
        y_fit,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        verbose=False,
    )
except CatBoostError:
    cpu_params = {k: v for k, v in fit_params.items() if k not in ["task_type", "devices"]}
    cpu_params["task_type"] = "CPU"
    model_es = CatBoostClassifier(**cpu_params)
    model_es.fit(
        X_fit,
        y_fit,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        verbose=False,
    )

best_iter = max(1, model_es.get_best_iteration())

final_params = {k: v for k, v in fit_params.items() if k not in ["od_type", "od_wait"]}
final_params["iterations"] = best_iter

clf_cb = CatBoostClassifier(**final_params)
clf_cb.fit(X_train_full, y, cat_features=cat_cols, verbose=False)

test_pred = clf_cb.predict_proba(X_test)[:, 1]

submit = sub.copy()
submit["Heart Disease"] = test_pred

if kaggle_dir.exists():
    submit.to_csv("submission.csv", index=False)
else:
    submit.to_csv("../data/submissions/catboost_baseline.csv", index=False)

print(f"Best iteration from holdout early stopping: {best_iter}")
submit.head()
